In [2]:
!pip install adamp

  Preparing metadata (setup.py) ... done
  Created wheel for adamp: filename=adamp-0.3.0-py3-none-any.whl size=5981 sha256=7090689a335399e5dc3ca64014e8c0c7cfde3f7f9954e81c87b4f0e1a6d4364f
  Stored in directory: /root/.cache/pip/wheels/33/f9/d6/b2ed816e1f321f6dcf72a99c954223b1259477095f40434979
Successfully built adamp


In [3]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import random
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ExponentialLR, CosineAnnealingWarmRestarts
from torch.optim import AdamW
from adamp import AdamP
from sklearn.model_selection import StratifiedKFold
import pytorch_lightning as pl
from pytorch_lightning import LightningModule
import transformers
from transformers import AutoModel, AutoTokenizer
transformers.logging.set_verbosity_error()

# Set up argument parser
import argparse
parser = argparse.ArgumentParser(description="AI_Text_Detector")
parser.add_argument('--pretrained_model', default='klue/roberta-large', type=str)  # Korean language model
parser.add_argument('--batch_size', default=8, type=int)
parser.add_argument('--lr', default=2e-5, type=float)
parser.add_argument('--epochs', default=5, type=int)
parser.add_argument('--max_length', default=512, type=int)  # Increased for article text
parser.add_argument('--train_data_path', default='./train.csv', type=str)
parser.add_argument('--test_data_path', default='./test.csv', type=str)
parser.add_argument('--optimizer', default='AdamW')
parser.add_argument('--lr_scheduler', default='cos')
parser.add_argument('--device', default='0', type=int)
parser.add_argument('--mixed_precision', default=16, type=int)
parser.add_argument('--cpu_workers', default=4, type=int)
parser.add_argument('--seed', default=516, type=int)
parser.add_argument('--date', default=20250520, type=int)
args = parser.parse_args('')

# 시드도 랜덤하게 넣어서 매번 학습 성능이 바뀌게 설
#def set_seeds(seed=args.seed):
 #   np.random.seed(seed)
  #  random.seed(seed)
   # torch.manual_seed(seed)
#    torch.cuda.manual_seed(seed)
 #   torch.backends.cudnn.deterministic = True
  #  torch.backends.cudnn.benchmark = False
   # pl.seed_everything(seed)
pl.seed_everything(None)
#set_seeds()
os.makedirs("saved", exist_ok=True)
os.makedirs("submission", exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

submission_id = f"{parser.description}_{args.date}"
print("Using PyTorch Ver", torch.__version__)
print("Using Lightning Ver", pl.__version__)
print("Fix Seed:", args.seed)
print("Submission ID:", submission_id)
print(f"Using model: {args.pretrained_model}")

# Load data
train_df = pd.read_csv('/kaggle/input/hallym-ai-text-classification/train.csv')
test_df = pd.read_csv('/kaggle/input/hallym-ai-text-classification/test.csv')
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")
print(f"Label distribution in train: {train_df['label'].value_counts().to_dict()}")

# Check for missing values
print(f"Missing values in train: {train_df.isnull().sum().sum()}")
print(f"Missing values in test: {test_df.isnull().sum().sum()}")

# Fill NaN values if any
train_df['text'] = train_df['text'].fillna("")
test_df['text'] = test_df['text'].fillna("")

# Prepare data
X = train_df["text"].values
y = train_df["label"].values
X_test = test_df["text"].values

print(f"X shape: {X.shape}, y shape: {y.shape}, X_test shape: {X_test.shape}")

# Custom dataset
class CustomDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = AutoTokenizer.from_pretrained(args.pretrained_model)
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        
        # Tokenize text
        encoded = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=args.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
            return_token_type_ids=True,
            return_attention_mask=True,
        )
        
        input_ids = encoded["input_ids"][0]
        token_type_ids = encoded["token_type_ids"][0]
        attention_mask = encoded["attention_mask"][0]
        
        if self.labels is not None:
            label = self.labels[idx]
            return [input_ids, token_type_ids, attention_mask], label
        else:
            return [input_ids, token_type_ids, attention_mask]

# Model definition
class SwiGLU(nn.Module):
    def forward(self, x):
        x, gate = x.chunk(2, dim=-1)
        return F.silu(gate) * x

class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.txt_model = AutoModel.from_pretrained(args.pretrained_model)
        
        # Get hidden dimensions from the model
        config = self.txt_model.config
        hidden_size = config.hidden_size
        
        # Text classifier with dropout for regularization
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size * 2),
            SwiGLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 1)
        )
    
    def forward(self, x):
        input_ids = x[0]
        token_type_ids = x[1]
        attention_mask = x[2]
        
        outputs = self.txt_model(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask,
        )
        
        # Get CLS token representation (the first token)
        txt_feature = outputs.last_hidden_state[:, 0, :]
        
        # Classification
        logits = self.classifier(txt_feature)
        
        return logits

class Model(LightningModule):
    def __init__(self, backbone, args):
        super().__init__()
        self.backbone = backbone
        self.args = args
        self.best_val_acc = 0.0
        
    def forward(self, x):
        return self.backbone(x)
    
    def step(self, batch):
        x, y = batch
        logits = self.backbone(x)
        loss = nn.BCEWithLogitsLoss()(logits.squeeze(), y.float())
        return loss, y, logits
    
    def training_step(self, batch, batch_idx):
        loss, y, logits = self.step(batch)
        preds = (torch.sigmoid(logits) > 0.5).float()
        accuracy = (preds.squeeze() == y).float().mean()
        
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_acc", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        loss, y, logits = self.step(batch)
        preds = (torch.sigmoid(logits) > 0.5).float()
        accuracy = (preds.squeeze() == y).float().mean()
        
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_acc", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        
        # Store predictions and labels for F1 calculation in on_validation_epoch_end
        if not hasattr(self, 'val_preds'):
            self.val_preds = []
            self.val_labels = []
            
        self.val_preds.append(preds.detach().cpu().numpy())
        self.val_labels.append(y.detach().cpu().numpy())
        
        return {"val_loss": loss, "val_acc": accuracy}
    
    # Use on_validation_epoch_end hook instead of validation_epoch_end (deprecated in PL 2.0+)
    def on_validation_epoch_end(self):
        from sklearn.metrics import f1_score
    
        if not hasattr(self, 'val_preds'):
            self.val_preds = []
            self.val_labels = []
    
        if len(self.val_preds) > 0:
            all_preds = np.concatenate(self.val_preds)
            all_truths = np.concatenate(self.val_labels)
    
            # f1_score 계산 시 average='macro' 옵션 추가
            f1 = f1_score(all_truths, all_preds, average='macro')
            self.log("val_f1", f1, prog_bar=True)
    
            if f1 > self.best_val_acc:
                self.best_val_acc = f1
    
        self.val_preds = []
        self.val_labels = []

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        if isinstance(batch, list) and len(batch) == 2:
            x, _ = batch  # In case we have labels
        else:
            x = batch
        logits = self.backbone(x)
        return torch.sigmoid(logits)
    
    def configure_optimizers(self):
        if self.args.optimizer == 'AdamW':
            optimizer = AdamW(self.parameters(), lr=self.args.lr, weight_decay=0.01)
        elif self.args.optimizer == 'AdamP':
            optimizer = AdamP(self.parameters(), lr=self.args.lr, weight_decay=0.01)
        
        if self.args.lr_scheduler == "none":
            return [optimizer]
        
        if self.args.lr_scheduler == 'cos':
            scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=1, T_mult=2)
        elif self.args.lr_scheduler == 'exp':
            scheduler = ExponentialLR(optimizer, gamma=0.9)
        
        return [optimizer], [scheduler]

# Training with K-Fold Cross-Validation
val_f1_list = []
preds_list = []

skf = StratifiedKFold(n_splits=5, shuffle=True)
for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"\n=== Fold {fold+1}/5 ===")
    
    X_train = X[train_index]
    X_val = X[val_index]
    y_train = y[train_index]
    y_val = y[val_index]
    
    print(f"Training samples: {len(X_train)}, Validation samples: {len(X_val)}")
    
    # Create datasets and dataloaders
    train_ds = CustomDataset(X_train, y_train)
    val_ds = CustomDataset(X_val, y_val)
    test_ds = CustomDataset(X_test, None)
    
    train_dataloader = DataLoader(
        train_ds, 
        batch_size=args.batch_size, 
        shuffle=True, 
        num_workers=args.cpu_workers
    )
    
    val_dataloader = DataLoader(
        val_ds, 
        batch_size=args.batch_size, 
        shuffle=False, 
        num_workers=args.cpu_workers
    )
    
    test_dataloader = DataLoader(
        test_ds, 
        batch_size=args.batch_size, 
        shuffle=False, 
        num_workers=args.cpu_workers
    )
    
    # Initialize model
    model = Model(Backbone(), args)
    
    # Setup callbacks
    callbacks = [
        pl.callbacks.ModelCheckpoint(
            dirpath="saved/", 
            filename=f"ai_detector_fold{fold}",
            monitor="val_f1", 
            mode="max",
            save_top_k=1,
            verbose=True
        ),
        pl.callbacks.EarlyStopping(
            monitor="val_f1",
            mode="max",
            patience=2,
            verbose=True
        )
    ]
    
    # Initialize trainer
    trainer = pl.Trainer(
        max_epochs=args.epochs, 
        accelerator="auto", 
        callbacks=callbacks,
        precision="16-mixed" if args.mixed_precision == 16 else args.mixed_precision,
        devices=[args.device],
        log_every_n_steps=10
    )
    
    # Train the model
    trainer.fit(model, train_dataloader, val_dataloader)
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    #폴드 끝나면 모델 삭제 후 메모리 비우기
    
    # Load best model
    best_model_path = trainer.checkpoint_callback.best_model_path
    if best_model_path:
        print(f"Loading best model from {best_model_path}")
        ckpt = torch.load(best_model_path)
        model.load_state_dict(ckpt['state_dict'])
    
    # Evaluate
    results = trainer.validate(model, dataloaders=val_dataloader)
    fold_f1 = trainer.callback_metrics.get("val_f1", 0.0).item()
    val_f1_list.append(fold_f1)
    print(f"Fold {fold+1} F1 Score: {fold_f1:.4f}")
    
    # Predict test data
    predictions = trainer.predict(model, dataloaders=test_dataloader)
    fold_preds = torch.cat(predictions).cpu().numpy()
    preds_list.append(fold_preds)

# Display validation results
val_f1_mean = np.mean(val_f1_list)
val_f1_std = np.std(val_f1_list)
print(f"\nCross-validation F1 Score: {val_f1_mean:.4f} ± {val_f1_std:.4f}")
for i, f1 in enumerate(val_f1_list):
    print(f"Fold {i+1}: {f1:.4f}")

# Average predictions across folds
final_preds = np.mean(preds_list, axis=0)
final_binary_preds = (final_preds > 0.5).astype(int)

# Prepare submission
submission = pd.read_csv('/kaggle/input/hallym-ai-text-classification/sample_submission.csv')
submission["label"] = final_binary_preds.squeeze()
submission_path = f"./submission/ai_detector_{args.pretrained_model.split('/')[-1]}_submit.csv"
submission.to_csv(submission_path, index=False)

print(f"\nPrediction Distribution: {np.bincount(final_binary_preds.squeeze().astype(int))}")
print(f"Submission saved to {submission_path}")

/usr/local/lib/python3.11/dist-packages/lightning_fabric/utilities/seed.py:42: No seed found, seed set to 0


Using PyTorch Ver 2.6.0+cu124
Using Lightning Ver 2.5.1.post0
Fix Seed: 516
Submission ID: AI_Text_Detector_20250520
Using model: klue/roberta-large
Train shape: (1500, 3), Test shape: (500, 2)
Label distribution in train: {0: 1250, 1: 250}
Missing values in train: 0
Missing values in test: 0
X shape: (1500,), y shape: (1500,), X_test shape: (500,)

=== Fold 1/5 ===
Training samples: 1200, Validation samples: 300


tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val_acc          │            1.0            │
│          val_f1           │            1.0            │
│         val_loss          │   0.009419907815754414    │
└───────────────────────────┴───────────────────────────┘

Fold 5 F1 Score: 1.0000


Predicting: |          | 0/? [00:00<?, ?it/s]


Cross-validation F1 Score: 0.9790 ± 0.0155
Fold 1: 0.9634
Fold 2: 0.9940
Fold 3: 0.9756
Fold 4: 0.9622
Fold 5: 1.0000

Prediction Distribution: [399 101]
Submission saved to ./submission/ai_detector_roberta-large_submit.csv


In [17]:
# Average predictions across folds (excluding Fold 2 = index 1)
selected_folds = [3, 1,4]
selected_preds = [preds_list[i] for i in selected_folds]
final_preds = np.mean(selected_preds, axis=0)
final_binary_preds = (final_preds > 0.5).astype(int)

# Prepare submission
submission = pd.read_csv('/kaggle/input/hallym-ai-text-classification/sample_submission.csv')
submission["label"] = final_binary_preds.squeeze()
submission_path = f"./submission/245_submit.csv"
submission.to_csv(submission_path, index=False)

print(f"\n[Fold 2,4,5 앙상블 결과]")
print(f"\nPrediction Distribution: {np.bincount(final_binary_preds.squeeze().astype(int))}")
print(f"Submission saved to {submission_path}")
#1,3,4,5폴드 앙상블


[Fold 2,4,5 앙상블 결과]

Prediction Distribution: [400 100]
Submission saved to ./submission/245_submit.csv
